In [ ]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [ ]:
train_texts = ds["train"]["text"]
train_labels = ds["train"]["label"]

test_texts = ds["test"]["text"]
test_labels = ds["test"]["label"]


In [ ]:
def tokenize(text):
    return text.lower().split()


In [ ]:
tokenized_train = [tokenize(text) for text in train_texts]
tokenized_test = [tokenize(text) for text in test_texts]


In [ ]:
from collections import Counter

MAX_VOCAB_SIZE = 20000
counter = Counter()

for tokens in tokenized_train:
    counter.update(tokens)

vocab = counter.most_common(MAX_VOCAB_SIZE)
word2idx = {word: idx + 2 for idx, (word, _) in enumerate(vocab)}

# Special tokens
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1


In [ ]:
def encode(tokens, word2idx):
    return [word2idx.get(token, word2idx["<UNK>"]) for token in tokens]

encoded_train = [encode(tokens, word2idx) for tokens in tokenized_train]
encoded_test = [encode(tokens, word2idx) for tokens in tokenized_test]


In [ ]:
from torch.nn.utils.rnn import pad_sequence
import torch

MAX_SEQ_LEN = 300

def pad_sequences(sequences, max_len):
    padded = []
    for seq in sequences:
        seq = seq[:max_len]
        padded.append(torch.tensor(seq))
    return pad_sequence(padded, batch_first=True, padding_value=0)

X_train = pad_sequences(encoded_train, MAX_SEQ_LEN)
X_test = pad_sequences(encoded_test, MAX_SEQ_LEN)

y_train = torch.tensor(train_labels)
y_test = torch.tensor(test_labels)


In [ ]:
import torch
import torch.nn as nn
class LSTMSentiment(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
    super().__init__()
    self.embedding = nn.Embedding(
        num_embeddings=vocab_size,
        embedding_dim=embed_dim,
        padding_idx=0
    )
    self.lstm = nn.LSTM(
        input_size=embed_dim,
        hidden_size=hidden_dim,
        num_layers=num_layers,
        batch_first=True,
        dropout=dropout if num_layers > 1 else 0
    )
    self.fc = nn.Linear(hidden_dim, 1)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    embedded = self.dropout(self.embedding(x))
    _, (hidden, _) = self.lstm(embedded)
    out = hidden[-1]
    return self.fc(out).squeeze(1)

In [ ]:
VOCAB_SIZE=len(word2idx)
EMBED_DIM=128
HIDDEN_DIM=256
NUM_LAYERS=2
DROPOUT=0.5
LR=1e-3
EPOCHS=5
BATCH_SIZE=64

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
train_dataset=TensorDataset(X_train,y_train)
test_dataset=TensorDataset(X_test,y_test)
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE)

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=LSTMSentiment(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)
criterion=nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=LR)

In [ ]:
def train_model(model,loader,optimizer,criterion):
  model.train()
  total_loss=0
  for X_batch, y_batch in loader:
      X_batch, y_batch = X_batch.to(device), y_batch.float().to(device)
      optimizer.zero_grad()
      outputs=model(X_batch)
      loss=criterion(outputs,y_batch)
      loss.backward()
      optimizer.step()
      total_loss+=loss.item()
  return total_loss/len(loader)

In [ ]:
from sklearn.metrics import accuracy_score
def evaluate_model(model,loader):
    model.eval()
    preds,labels=[],[]
    with torch.no_grad():
      for X_batch, y_batch in loader:
          X_batch=X_batch.to(device)
          outputs=model(X_batch)
          predictions=torch.sigmoid(outputs)>0.5
          preds.extend(predictions.cpu().numpy())
          labels.extend(y_batch.numpy())
    return accuracy_score(labels,preds)

In [ ]:
MAX_SEQ_LEN = 150


In [ ]:
HIDDEN_DIM = 128
NUM_LAYERS = 1


In [ ]:
BATCH_SIZE = 128


In [ ]:
import torch
import torch.nn as nn
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score
from datasets import load_dataset

# --- Data Preprocessing (Consolidated) ---

ds = load_dataset("stanfordnlp/imdb")
train_texts = list(ds["train"]["text"])
train_labels = ds["train"]["label"]

test_texts = list(ds["test"]["text"])
test_labels = list(ds["test"]["label"])

def tokenize(text):
    return text.lower().split()

tokenized_train = [tokenize(text) for text in train_texts]
tokenized_test = [tokenize(text) for text in test_texts]

MAX_VOCAB_SIZE = 20000
counter = Counter()
for tokens in tokenized_train:
    counter.update(tokens)
vocab = counter.most_common(MAX_VOCAB_SIZE)
word2idx = {word: idx + 2 for idx, (word, _) in enumerate(vocab)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1

def encode(tokens, word2idx):
    return [word2idx.get(token, word2idx["<UNK>"]) for token in tokens]

encoded_train = [encode(tokens, word2idx) for tokens in tokenized_train]
encoded_test = [encode(tokens, word2idx) for tokens in tokenized_test]

MAX_SEQ_LEN = 150 # Updated from kernel state

def pad_sequences(sequences, max_len):
    padded = []
    for seq in sequences:
        seq = seq[:max_len]
        padded.append(torch.tensor(seq))
    return pad_sequence(padded, batch_first=True, padding_value=0)

X_train = pad_sequences(encoded_train, MAX_SEQ_LEN)
X_test = pad_sequences(encoded_test, MAX_SEQ_LEN)
y_train = torch.tensor(train_labels)
y_test = torch.tensor(test_labels)

BATCH_SIZE = 128 # Updated from kernel state
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# --- Model Definition ---

class LSTMSentiment(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout):
    super().__init__()
    self.embedding = nn.Embedding(
        num_embeddings=vocab_size,
        embedding_dim=embed_dim,
        padding_idx=0
    )
    self.lstm = nn.LSTM(
        input_size=embed_dim,
        hidden_size=hidden_dim,
        num_layers=num_layers,
        batch_first=True,
        dropout=dropout if num_layers > 1 else 0
    )
    self.fc = nn.Linear(hidden_dim, 1)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    embedded = self.dropout(self.embedding(x))
    _, (hidden, _) = self.lstm(embedded)
    out = hidden[-1]
    return self.fc(out).squeeze(1)

# --- Training and Evaluation Functions ---

def train_model(model, loader, optimizer, criterion):
  model.train()
  total_loss=0
  for X_batch, y_batch in loader:
      X_batch, y_batch = X_batch.to(device), y_batch.float().to(device)
      optimizer.zero_grad()
      outputs=model(X_batch)
      loss=criterion(outputs,y_batch)
      loss.backward()
      optimizer.step()
      total_loss+=loss.item()
  return total_loss/len(loader)

def evaluate_model(model, loader):
    model.eval()
    preds,labels=[],[]
    with torch.no_grad():
      for X_batch, y_batch in loader:
          X_batch=X_batch.to(device)
          outputs=model(X_batch)
          predictions=torch.sigmoid(outputs)>0.5
          preds.extend(predictions.cpu().numpy())
          labels.extend(y_batch.numpy())
    return accuracy_score(labels,preds)

# --- Model Parameters and Initialization ---

VOCAB_SIZE = len(word2idx)
EMBED_DIM = 128
HIDDEN_DIM = 128 # Updated from kernel state
NUM_LAYERS = 1   # Updated from kernel state
DROPOUT = 0.5
LR = 1e-3
EPOCHS = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMSentiment(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# --- Training Loop ---

for epoch in range(EPOCHS):
    train_loss = train_model(model, train_loader, optimizer, criterion)

    if (epoch + 1) % 2 == 0:
        test_acc = evaluate_model(model, test_loader)
        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"Training Loss: {train_loss:.4f}")
        print(f"Test Accuracy: {test_acc:.4f}\n")

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Epoch 2/5
Training Loss: 0.6886
Test Accuracy: 0.5391

Epoch 4/5
Training Loss: 0.6859
Test Accuracy: 0.5269



In [ ]:
train_subset=TensorDataset(X_train[:5000],y_train[:5000])
train_loader=DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
!pip install fastai


In [ ]:
from fastai.text.all import *


In [ ]:
path=untar_data(URLs.IMDB)

In [ ]:
dls=TextDataLoaders.from_folder(
    path,
    train='train',
    valid='test',
    bs=64,
    seq_len=150
)

In [ ]:
learn=text_classifier_learner(
    dls,
    AWD_LSTM,
    drop_mult=0.5,
    metrics=accuracy
)

In [ ]:
learn.fit_one_cycle(1,2e-2)

epoch,train_loss,valid_loss,accuracy,time
0,0.427329,0.376692,0.832560,3:48:36


In [ ]:
!pip install transformers datasets


In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader

In [ ]:
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")
def tokenize_batch(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

In [ ]:
import torch
class IMDBDataset(torch.utils.data.Dataset):
  def __init__(self,texts,labels):
    self.encodings=tokenize_batch(texts)
    self.labels=torch.tensor(labels)
  def __getitem__(self,idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item["labels"]=self.labels[idx]
    return item
  def __len__(self):
    return len(self.labels)

In [ ]:
train_dataset=IMDBDataset(train_texts,train_labels)
test_dataset=IMDBDataset(test_texts,test_labels)
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=16)

In [ ]:
model=BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
from torch.optim import AdamW
optimizer=AdamW(model.parameters(),lr=2e-5)
model.train()
for epoch in range(2):
  for batch in train_loader:
    optimizer.zero_grad()
    batch={k:v.to(device) for k,v in batch.items()}
    outputs=model(**batch)
    loss=outputs.loss
    loss.backward()
    optimizer.step()
  print(f"Epoch{epoch+1} completed")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_bert(model, dataloader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = outputs.logits.argmax(dim=1)
            preds.extend(predictions.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


In [ ]:
!pip install -q transformers datasets accelerate
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [ ]:
dataset=load_dataset("imdb")
train_texts=dataset["train"]["text"]
train_labels=dataset['train']["label"]
test_texts=dataset["test"]["text"]
test_labels=dataset["test"]["label"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
tokenizer=BertTokenizerFast.from_pretrained("bert-base-uncased")
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )
tokenized_train=dataset["train"].map(tokenize,batched=True)
tokenized_test=dataset["test"].map(tokenize,batched=True)
tokenized_test.set_format("torch",columns=["input_ids","attention_mask","label"])
tokenized_train.set_format("torch",columns=["input_ids","attention_mask","label"])


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
model=BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def compute_metrics(eval_pred):
    logits,labels=eval_pred
    preds=np.argmax(logits, axis=1)
    precision,recall,f1,_=precision_recall_fscore_support(
        labels,preds,average="binary"
    )
    acc=accuracy_score(labels,preds)
    return{
        "accuracy":acc,
        "precision":precision,
        "recall":recall,
        "f1":f1
    }

In [ ]:
training_args=TrainingArguments(
    output_dir="./bert-imdb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    metric_for_best_model="f1",
    report_to="none"
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()
results=trainer.evaluate()
print(results)

/tmp/ipython-input-463321333.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer=Trainer(


Step,Training Loss
500,0.333200
1000,0.257800
1500,0.242900
2000,0.166000
2500,0.154200
3000,0.157700
3500,0.097900
4000,0.093300
4500,0.089100


{'eval_loss': 0.3214673101902008, 'eval_accuracy': 0.92448, 'eval_precision': 0.9237342277591439, 'eval_recall': 0.92536, 'eval_f1': 0.9245463991687315, 'eval_runtime': 351.5103, 'eval_samples_per_second': 71.122, 'eval_steps_per_second': 4.447, 'epoch': 3.0}


In [ ]:
from datasets import load_dataset
dataset=load_dataset("imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
from transformers import BertTokenizerFast
tokenizer=BertTokenizerFast.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
def tokenize_text(batch):
  return tokenizer(
      batch["text"],
      padding="max_length",
      truncation=True,
      max_length=256
  )

In [ ]:
tokenized_train=dataset["train"].map(tokenize_text,batched=True)
tokenized_test=dataset["test"].map(tokenize_text,batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
tokenized_train.set_format(
    type="torch",
    columns=["input_ids","attention_mask","label"]
)
tokenized_test.set_format(
    type="torch",
    columns=["input_ids","attention_mask","label"]
)

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizerFast

# Re-loading dataset and tokenizer as they were not defined in current session
dataset = load_dataset("imdb")
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize_text(batch):
  return tokenizer(
      batch["text"],
      padding="max_length",
      truncation=True,
      max_length=256
  )

tokenized_train = dataset["train"].map(tokenize_text, batched=True)
tokenized_test = dataset["test"].map(tokenize_text, batched=True)

tokenized_train.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)
tokenized_test.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)


first_item = tokenized_train[0]
input_ids = first_item["input_ids"]
attention_mask = first_item["attention_mask"]
label = first_item["label"] # Corrected to use 'label' (singular)

sample = {
    "input_ids": input_ids,
    "attention_mask": attention_mask,
    "labels": label # Use 'labels' as the key to match common practice and the dataset
}

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
"""{"input_ids":Tensor[256],
 "attention_mask":Tensor[256],
 "label":0 or 1}"""

'{"input_ids":Tensor[256],\n "attention_mask":Tensor[256],\n "label":0 or 1}'

In [ ]:
display(tokenized_train[:3])

{'label': tensor([0, 0, 0]),
 'input_ids': tensor([[  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
           2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
           2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
           2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
           1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
           2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
           6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
           1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
           5436,  2003,  8857,  2105,  1037,  2402,  4467,  3689,  3076,  2315,
          14229,  2040,  4122,  2000,  4553,  2673,  2016,  2064,  2055,  2166,
           1012,  1999,  3327,  2016,  4122,  2000,  3579,  2014,  3086,  2015,
           2000,  2437,  2070,  4066,  1997,  4516,  2006,  2054,  1996,  2779

In [ ]:
process_bert_input(example_data)

input_ids type: <class 'torch.Tensor'>, shape: torch.Size([256])
attention_mask type: <class 'torch.Tensor'>, shape: torch.Size([256])
label type: <class 'int'>, value: 0


In [ ]:
import torch
from typing import Dict, Union

def process_bert_input(data: Dict[str, Union[torch.Tensor, int]]) -> None:
    """
    Processes input dictionary for BERT model.
    Expected structure:
    {
        "input_ids": torch.Tensor # shape: [sequence_length] e.g., [256]
        "attention_mask": torch.Tensor # shape: [sequence_length] e.g., [256]
        "label": int # value: 0 or 1
    }
    """
    print(f"input_ids type: {type(data['input_ids'])}, shape: {data['input_ids'].shape}")
    print(f"attention_mask type: {type(data['attention_mask'])}, shape: {data['attention_mask'].shape}")
    print(f"label type: {type(data['label'])}, value: {data['label']}")


# Example of actual data that would fit this description
example_data = {
    "input_ids": torch.randint(0, tokenizer.vocab_size, (256,)),
    "attention_mask": torch.ones(256, dtype=torch.long),
    "label": 0
}

# You can then pass this actual data to a function that expects this structure
# process_bert_input(example_data)

# If you just wanted to have a comment next to your cell, you could do:
# Expected dictionary structure:
# {
#  "input_ids": torch.Tensor[256],
#  "attention_mask": torch.Tensor[256],
#  "label": 0 or 1
# }

In [ ]:
import torch
import torch.nn as nn
class LSTMSentimentClassifier(nn.Module):
  def __init__(
      self,
      vocab_size,
      embedding_dim,
      hidden_dim,
      output_dim,
      num_layers=1,
      bidirectional=False,
      dropout=0.5,
      pad_idx=0
  ):
      super().__init__()
      self.embedding=nn.Embedding(
          vocab_size,
          embedding_dim,padding_idx=pad_idx
      )
      self.lstm=nn.LSTM(
          embedding_dim,
          hidden_dim,
          num_layers=num_layers,
          bidirectional=bidirectional,
          batch_first=True,
          dropout=dropout if num_layers>1 else 0.0
      )
      lstm_output_dim=hidden_dim*(2 if bidirectional else 1)
      self.fc=nn.Linear(lstm_output_dim,output_dim)
      self.dropout=nn.Dropout(dropout)

  def forward(self, input_ids):
      # input_ids: (batch_size,seq_len)
      embedded=self.dropout(self.embedding(input_ids))
      outputs,(hidden,cell)=self.lstm(embedded)

      if self.lstm.bidirectional:
          # Concatenate the last two hidden states for bidirectional LSTM
          hidden=torch.cat((hidden[-2],hidden[-1]),dim=1)
      else:
          # Take the last hidden state for unidirectional LSTM
          hidden=hidden[-1]

      hidden=self.dropout(hidden)
      logits=self.fc(hidden)
      return logits

In [ ]:
import torch
import torch.nn as nn
from collections import Counter
from datasets import load_dataset

# --- Data Preprocessing (minimum to get word2idx and VOCAB_SIZE) ---
ds = load_dataset("stanfordnlp/imdb")
train_texts = list(ds["train"]["text"])

def tokenize(text):
    return text.lower().split()

tokenized_train = [tokenize(text) for text in train_texts]

MAX_VOCAB_SIZE = 20000
counter = Counter()
for tokens in tokenized_train:
    counter.update(tokens)
vocab = counter.most_common(MAX_VOCAB_SIZE)
word2idx = {word: idx + 2 for idx, (word, _) in enumerate(vocab)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1

# --- Model Parameters and Initialization ---
VOCAB_SIZE = len(word2idx)
EMBED_DIM = 128
HIDDEN_DIM = 128 # Using the updated value from previous notebook state
NUM_LAYERS = 1   # Using the updated value from previous notebook state
DROPOUT = 0.5    # From previous notebook state

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMSentimentClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=1, # Binary classification
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=2e-3)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
def train_model(model,dataLoader,optimizer,criterion,device):
    model.train()
    total_loss=0
    for batch in dataloader:
        input_ids=batch["input_ids"].to(device)
        labels=batch["labels"].to(device)
        optimizer.zero_grad()
        outputs=model(input_ids)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(dataloader)

In [ ]:
def evaluate_model(model,dataloader,device):
    model.eval()
    correct=0
    total=0
    with torch.no_grad():
      for batch in dataloader:
          input_ids=batch["input_ids"].to(device)
          labels=batch["labels"].to(device)
          outputs=model(input_ids)
          predictions=outputs.argmax(dim=1)
          correct+=(predictions==labels).sum().item()
          total+=labels.size(0)
    return correct/total

In [ ]:
VOCAB_SIZE=30000
EMBEDDING_DIM=300
HIDDEN_DIM=256
OUTPUT_DIM=2
PAD_IDX=0
model=LSTMSentimentClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    num_layers=2,
    bidirectional=True,
    dropout=0.5,
    pad_idx=PAD_IDX
)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

LSTMSentimentClassifier(
  (embedding): Embedding(30000, 300, padding_idx=0)
  (lstm): LSTM(300, 256, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=512, out_features=2, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [ ]:
import torch
from transformers import BertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.to(device)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset

# Re-define tokenizer, tokenize_batch, IMDBDataset, and DataLoaders
# to ensure they are available in the current execution context.
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_batch(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

class IMDBDataset(torch.utils.data.Dataset):
  def __init__(self,texts,labels):
    self.encodings=tokenize_batch(texts)
    self.labels=torch.tensor(labels)
  def __getitem__(self,idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item["labels"]=self.labels[idx]
    return item
  def __len__(self):
    return len(self.labels)

# Load dataset again to ensure train_texts and train_labels are available
dataset = load_dataset("imdb")
train_texts = list(dataset["train"]["text"])
train_labels = list(dataset['train']["label"])
test_texts = list(dataset["test"]["text"])
test_labels = list(dataset["test"]["label"])

train_dataset=IMDBDataset(train_texts,train_labels)
test_dataset=IMDBDataset(test_texts,test_labels)
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=16)

# Initialize the model
model=BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

total_steps = len(train_loader) * 3  # 3 epochs recommended for BERT

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- Tokenizer and Dataset preparation (copied from pkj2LyqSrOGT) ---
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_batch(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

class IMDBDataset(torch.utils.data.Dataset):
  def __init__(self,texts,labels):
    self.encodings=tokenize_batch(texts)
    self.labels=torch.tensor(labels)
  def __getitem__(self,idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item["labels"]=self.labels[idx]
    return item
  def __len__(self):
    return len(self.labels)

# Load dataset and prepare data loaders (copied from pkj2LyqSrOGT)
dataset = load_dataset("imdb")
train_texts = list(dataset["train"]["text"])
train_labels = list(dataset['train']["label"])
test_texts = list(dataset["test"]["text"])
test_labels = list(dataset["test"]["label"])

train_dataset=IMDBDataset(train_texts,train_labels)
test_dataset=IMDBDataset(test_texts,test_labels)
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=16)

# --- Model Initialization (copied from pkj2LyqSrOGT) ---
model=BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# --- Optimizer and Scheduler Setup (copied from pkj2LyqSrOGT) ---
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

EPOCHS = 3 # Defined here for clarity
total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# --- Training and Evaluation Functions ---
def train_bert(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_bert(model, dataloader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = outputs.logits.argmax(dim=1)
            preds.extend(predictions.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# --- Training Loop ---
for epoch in range(EPOCHS):
  train_loss=train_bert(
      model,train_loader,optimizer,scheduler,device)
  test_metrics=evaluate_bert(
      model,test_loader)
  print(f"Epoch {epoch+1}/{EPOCHS}")
  print(f"Train Loss:{train_loss:.4f}")
  print(f"Test Accuracy:{test_metrics['accuracy']:.4f}")
  print(f"Test Precision:{test_metrics['precision']:.4f}")
  print(f"Test Recall:{test_metrics['recall']:.4f}")
  print(f"Test F1 Score:{test_metrics['f1']:.4f}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3
Train Loss:0.3788
Test Accuracy:0.8806
Test Precision:0.8476
Test Recall:0.9282
Test F1 Score:0.8861
Epoch 2/3
Train Loss:0.1893
Test Accuracy:0.8865
Test Precision:0.8599
Test Recall:0.9234
Test F1 Score:0.8905
Epoch 3/3
Train Loss:0.0836
Test Accuracy:0.8896
Test Precision:0.8869
Test Recall:0.8932
Test F1 Score:0.8900


In [ ]:
import torch
from transformers import BertForSequenceClassification
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
for param in model.bert.parameters():
    param.requires_grad=False

In [ ]:
import torch
from transformers import BertForSequenceClassification
from torch.optim import AdamW

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
model.to(device)

for param in model.bert.parameters():
    param.requires_grad=False

optimizer=AdamW(
    filter(lambda p: p.requires_grad,model.parameters()),
    lr=1e-3,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def train_classifier_head(model,dataloader,optimizer,device):
    model.train()
    total_loss=0
    for batch in dataloader:
        input_ids=batch["input_ids"].to(device)
        attention_mask=batch["attention_mask"].to(device) # Corrected: use attention_mask from batch
        labels=batch["labels"].to(device)
        optimizer.zero_grad()
        outputs=model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss=outputs.loss
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(dataloader)


In [ ]:
def evaluate_classifier(model,dataloader,device):
    model.eval()
    correct,total=0,0
    with torch.no_grad():
        for batch in dataloader:
            input_ids=batch["input_ids"].to(device)
            attention_mask=batch["attention_mask"].to(device)
            labels=batch["labels"].to(device)
            outputs=model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            predictions=torch.argmax(outputs.logits, dim=1)
            correct+=(predictions==labels).sum().item()
            total+=labels.size(0)
    return correct/total

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset

# --- Tokenizer and Dataset preparation ---
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_batch(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

class IMDBDataset(torch.utils.data.Dataset):
  def __init__(self,texts,labels):
    self.encodings=tokenize_batch(texts)
    self.labels=torch.tensor(labels)
  def __getitem__(self,idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item["labels"]=self.labels[idx]
    return item
  def __len__(self):
    return len(self.labels)

# Load dataset
dataset = load_dataset("imdb")
train_texts = list(dataset["train"]["text"])
train_labels = list(dataset['train']["label"])
test_texts = list(dataset["test"]["text"])
test_labels = list(dataset["test"]["label"])

train_dataset=IMDBDataset(train_texts,train_labels)
test_dataset=IMDBDataset(test_texts,test_labels)
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=16)

# --- Model Initialization and Optimizer Setup ---
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
model.to(device)

for param in model.bert.parameters():
    param.requires_grad=False

optimizer=AdamW(
    filter(lambda p: p.requires_grad,model.parameters()),
    lr=1e-3,
)

EPOCHS=3

def train_classifier_head(model,dataloader,optimizer,device):
    model.train()
    total_loss=0
    for batch in dataloader:
        input_ids=batch["input_ids"].to(device)
        attention_mask=batch["attention_mask"].to(device)
        labels=batch["labels"].to(device)
        optimizer.zero_grad()
        outputs=model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss=outputs.loss
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(dataloader)

def evaluate_classifier(model,dataloader,device):
    model.eval()
    correct,total=0,0
    with torch.no_grad():
        for batch in dataloader:
            input_ids=batch["input_ids"].to(device)
            attention_mask=batch["attention_mask"].to(device)
            labels=batch["labels"].to(device)
            outputs=model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            predictions=torch.argmax(outputs.logits, dim=1)
            correct+=(predictions==labels).sum().item()
            total+=labels.size(0)
    return correct/total

for epoch in range(EPOCHS):
    train_loss=train_classifier_head(
        model,train_loader,optimizer,device)
    test_acc=evaluate_classifier(
        model, test_loader,device
    )
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss:{train_loss:.4f}")
    print(f"Test Accuracy:{test_acc:.4f}\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
